In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IVGMM
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. LOAD MASTER DATA & DEFINE BLOCS
# ==============================================================================
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
master_file = PROCESSED_DIR / "master_panel_data.csv"

df = pd.read_csv(master_file)
df = df.set_index(['Country Name', 'Year']).sort_index()

BRICS_NATIONS = ["Brazil", "Russia", "India", "China", "Egypt", "United Arab Emirates"]
OECD_EMERGING = ["Chile", "Colombia", "Costa Rica", "Turkey", "Hungary", "Poland"]

def categorize_economy(country):
    if country in BRICS_NATIONS: return 'BRICS (Developing)'
    elif country in OECD_EMERGING: return 'OECD (Emerging)'
    else: return 'OECD (Advanced)'

df['Economic_Bloc'] = [categorize_economy(c) for c, y in df.index]

Y_VAR = 'GDP growth (annual %)'
DEBT_VAR = 'General Government Debt (% of GDP)'
EXOG_VARS = ['Gross capital formation (% of GDP)', 'Inflation (Annual %)', 'Population growth (annual %)']

# ==============================================================================
# 2. CONSTRUCT GMM LAGS & EXCEL FORMATTER
# ==============================================================================
df['Inst_GDP_Lag2'] = df.groupby(level=0)[Y_VAR].shift(2)
df['Inst_Debt_Lag2'] = df.groupby(level=0)[DEBT_VAR].shift(2)
df['Inst_GDP_Lag3'] = df.groupby(level=0)[Y_VAR].shift(3)
df['Inst_Debt_Lag3'] = df.groupby(level=0)[DEBT_VAR].shift(3)

df[f'D_{Y_VAR}'] = df.groupby(level=0)[Y_VAR].diff(1)
df[f'D_{DEBT_VAR}'] = df.groupby(level=0)[DEBT_VAR].diff(1)
for var in EXOG_VARS: df[f'D_{var}'] = df.groupby(level=0)[var].diff(1)

df[f'D_{Y_VAR}_Lag1'] = df.groupby(level=0)[f'D_{Y_VAR}'].shift(1)


def print_excel_summary(res, model_title):
    """Parses linearmodels output into a strict Excel Solver format."""
    print(f"\nSUMMARY OUTPUT: {model_title}")
    print("\nRegression Statistics")
    # GMM does not calculate R2 or SSR natively like OLS
    print(f"{'Multiple R':<20} #N/A")
    print(f"{'R Square':<20} {getattr(res, 'rsquared', '#N/A')}")
    print(f"{'Adjusted R Square':<20} #N/A")
    print(f"{'Standard Error':<20} #N/A")
    print(f"{'Observations':<20} {res.nobs}")
    
    print("\nANOVA")
    print("ANOVA not mathematically applicable for GMM instrumental estimation.")
    
    print("\n" + "-"*120)
    print(f"{'':<40} {'Coefficients':<15} {'Standard Error':<15} {'t Stat':<10} {'P-value':<10} {'Lower 95%':<12} {'Upper 95%'}")
    
    params, se, tstat, pval, ci = res.params, res.std_errors, res.tstats, res.pvalues, res.conf_int()
    for var in params.index:
        c, s, t, p = params[var], se[var], tstat[var], pval[var]
        l, u = ci.loc[var, 'lower'], ci.loc[var, 'upper']
        print(f"{var[:38]:<40} {c:<15.6f} {s:<15.6f} {t:<10.6f} {p:<10.6f} {l:<12.6f} {u:<12.6f}")
    print("-" * 120 + "\n")

# ==============================================================================
# 3. GMM MODEL EXECUTION
# ==============================================================================
def run_dynamic_gmm(data, bloc_name):
    model_vars = [f'D_{Y_VAR}', f'D_{Y_VAR}_Lag1', f'D_{DEBT_VAR}'] + \
                 [f'D_{v}' for v in EXOG_VARS] + \
                 ['Inst_GDP_Lag2', 'Inst_Debt_Lag2', 'Inst_GDP_Lag3', 'Inst_Debt_Lag3']
                 
    reg_data = data[model_vars].dropna()
    if len(reg_data) < 30: return
    
    Y = reg_data[f'D_{Y_VAR}']
    X_exog = sm.add_constant(reg_data[[f'D_{v}' for v in EXOG_VARS]])
    X_endog = reg_data[[f'D_{Y_VAR}_Lag1', f'D_{DEBT_VAR}']]
    Instruments = reg_data[['Inst_GDP_Lag2', 'Inst_Debt_Lag2', 'Inst_GDP_Lag3', 'Inst_Debt_Lag3']]
    
    gmm_model = IVGMM(Y, X_exog, X_endog, Instruments, weight_type='robust')
    gmm_res = gmm_model.fit()
    
    j_stat = gmm_res.j_stat
    print(f"\n[HANSEN J-TEST] Stat: {j_stat.stat:.2f} | p-value: {j_stat.pval:.4f} -> {'VALID' if j_stat.pval > 0.05 else 'INVALID'} Instruments")
    
    print_excel_summary(gmm_res, model_title=f"DIFFERENCE GMM: {bloc_name}")

if __name__ == "__main__":
    for bloc in ['OECD (Advanced)', 'OECD (Emerging)', 'BRICS (Developing)']:
        df_bloc = df[df['Economic_Bloc'] == bloc]
        run_dynamic_gmm(df_bloc, bloc)


[HANSEN J-TEST] Stat: 2.48 | p-value: 0.2897 -> VALID Instruments

SUMMARY OUTPUT: DIFFERENCE GMM: OECD (Advanced)

Regression Statistics
Multiple R           #N/A
R Square             -0.14066935355156684
Adjusted R Square    #N/A
Standard Error       #N/A
Observations         770

ANOVA
ANOVA not mathematically applicable for GMM instrumental estimation.

------------------------------------------------------------------------------------------------------------------------
                                         Coefficients    Standard Error  t Stat     P-value    Lower 95%    Upper 95%
const                                    0.119131        0.259724        0.458686   0.646460   -0.389917    0.628180    
D_Gross capital formation (% of GDP)     0.263318        0.189215        1.391636   0.164033   -0.107536    0.634173    
D_Inflation (Annual %)                   -0.189198       0.132445        -1.428506  0.153146   -0.448786    0.070389    
D_Population growth (annual %)       